# Step 2: Data Cleaning & Preprocessing

Standardizes all audio files to:
- **Uniform Sampling Rate:** 16,000 Hz
- **Channel Format:** Mono (single channel)
- **Fixed Duration / Equalization:** 4.0 seconds (64,000 samples @ 16 kHz) via zero-padding or truncation
- **Amplitude Normalization:** Peak normalization to $[-0.95, 0.95]$
- **Balanced Dataset:** Generates `manifest.csv` with 1,500 Real and 1,500 Fake samples

In [1]:
import os
import json
import random
from pathlib import Path
import numpy as np
import soundfile as sf
import torch
import torchaudio
import pandas as pd
from tqdm import tqdm

BASE_DIR = Path.cwd().parent if Path.cwd().name == "src" else Path.cwd()
DATA_RAW_DIR = BASE_DIR / "data" / "raw"
DATA_PROCESSED_DIR = BASE_DIR / "data" / "processed_clean"
LOGS_DIR = BASE_DIR / "logs"

SAMPLE_RATE = 16000
TARGET_DURATION = 4.0
TARGET_SAMPLES = int(SAMPLE_RATE * TARGET_DURATION)  # 64,000
NORM_PEAK = 0.95
TARGET_PER_CLASS = 1500

### Preprocessing Routine

In [2]:
def process_single_audio(file_path: Path, output_path: Path, target_sr=16000, target_samples=64000, norm_peak=0.95):
    try:
        data, sr = sf.read(str(file_path), dtype="float32")
        orig_duration = len(data) / sr

        # 1. Convert to Mono
        if data.ndim > 1:
            data = np.mean(data, axis=1)

        # 2. Resample to 16 kHz
        if sr != target_sr:
            tensor_data = torch.from_numpy(data).unsqueeze(0)
            resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=target_sr)
            data = resampler(tensor_data).squeeze(0).numpy()

        # 3. Length Equalization (fixed 4.0s = target_samples)
        if len(data) < target_samples:
            data = np.pad(data, (0, target_samples - len(data)), mode="constant")
        elif len(data) > target_samples:
            data = data[:target_samples]

        # 4. Amplitude Normalization
        max_abs = np.max(np.abs(data))
        if max_abs > 1e-6:
            data = (data / max_abs) * norm_peak

        output_path.parent.mkdir(parents=True, exist_ok=True)
        sf.write(str(output_path), data, target_sr, subtype="PCM_16")

        return {
            "status": "success",
            "file_name": file_path.name,
            "orig_path": str(file_path),
            "proc_path": str(output_path),
            "orig_sr": sr,
            "orig_duration": orig_duration,
            "proc_duration": len(data) / target_sr,
            "samples": len(data)
        }
    except Exception as e:
        return {"status": "failed", "file_name": file_path.name, "orig_path": str(file_path), "error": str(e)}


In [3]:
# Run Preprocessing
raw_fake_dir = DATA_RAW_DIR / "FAKE"
raw_real_dir = DATA_RAW_DIR / "Real_audio"
valid_exts = {".wav", ".flac"}

fake_candidates = [f for f in raw_fake_dir.glob("*.*") if f.suffix.lower() in valid_exts]
real_candidates = [f for f in raw_real_dir.glob("*.*") if f.suffix.lower() in valid_exts]
random.seed(42)
random.shuffle(fake_candidates)
random.shuffle(real_candidates)

out_fake_dir = DATA_PROCESSED_DIR / "FAKE"
out_real_dir = DATA_PROCESSED_DIR / "Real_audio"
manifest_records = []
failed_records = []

# Process Fake class
fake_count = 0
for f in tqdm(fake_candidates, desc="Processing Fake"):
    if fake_count >= TARGET_PER_CLASS:
        break
    res = process_single_audio(f, out_fake_dir / f"{f.stem}.wav", SAMPLE_RATE, TARGET_SAMPLES, NORM_PEAK)
    if res["status"] == "success":
        res["label"], res["label_id"] = "Fake", 1
        manifest_records.append(res)
        fake_count += 1
    else:
        failed_records.append(res)

# Process Real class
real_count = 0
for f in tqdm(real_candidates, desc="Processing Real"):
    if real_count >= TARGET_PER_CLASS:
        break
    res = process_single_audio(f, out_real_dir / f"{f.stem}.wav", SAMPLE_RATE, TARGET_SAMPLES, NORM_PEAK)
    if res["status"] == "success":
        res["label"], res["label_id"] = "Real", 0
        manifest_records.append(res)
        real_count += 1
    else:
        failed_records.append(res)

manifest_df = pd.DataFrame(manifest_records)
manifest_path = DATA_PROCESSED_DIR / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)

print(f"\nPreprocessing Complete! Saved {len(manifest_df):,} files into {DATA_PROCESSED_DIR}")
print(manifest_df["label"].value_counts())


Processing Real:   9%|▉         | 2941/31405 [00:11<01:50, 258.64it/s]



Preprocessing Complete! Saved 3,000 files into e:\Deepfake\data\processed_clean
label
Fake    1500
Real    1500
Name: count, dtype: int64
